# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [7]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [8]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/customer"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 3

--- Customer_20260207170555.parquet ---
  Registros: 2
  Colunas: ['raw_value']
  ❌ Não contém dados JSON

--- Customer_20260325043100.parquet ---
  Registros: 10
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']

--- placeholder_customer_20260324.parquet ---
  Registros: 1
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [9]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema)).select("parsed.*")
        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 2

Comparando: Customer_20260325043100.parquet vs placeholder_customer_20260324.parquet
  ✅ Mesmas colunas (34 colunas)
  ✅ Tipos de dados idênticos

✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!


In [10]:
from pyspark.sql.functions import from_json, schema_of_json, col
from pyspark.sql.types import StructType, StringType

def flatten_df(df, sep="_"):
    """Achata recursivamente colunas StructType em colunas simples."""
    colunas_complexas = True
    while colunas_complexas:
        colunas_complexas = False
        novas_colunas = []
        for field in df.schema.fields:
            if isinstance(field.dataType, StructType):
                colunas_complexas = True
                for sub in field.dataType.fields:
                    novo_nome = f"{field.name}{sep}{sub.name}"
                    novas_colunas.append(df[field.name][sub.name].alias(novo_nome))
            else:
                novas_colunas.append(df[field.name])
        df = df.select(novas_colunas)
    return df

def parse_nested_json(df, sep="_", max_depth=3):
    """Detecta e parseia colunas string que contêm JSON, depois achata."""
    for _ in range(max_depth):
        str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        if not str_cols:
            break

        amostra = df.select(*[df[c] for c in str_cols]).filter(
            df[str_cols[0]].isNotNull()
        ).first()
        if not amostra:
            break

        cols_json = []
        for i, c in enumerate(str_cols):
            val = amostra[i]
            if val and isinstance(val, str) and val.strip() and val.strip()[0] in ('{', '['):
                cols_json.append((c, val))

        if not cols_json:
            break

        for c, val in cols_json:
            schema = schema_of_json(val)
            df = df.withColumn(c, from_json(df[c], schema))

        df = flatten_df(df, sep)
    return df

# Para cada arquivo com JSON, parsear, expandir sub-JSONs e achatar
for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"--- {nome}: Sem JSON, mostrando dados diretos ---")
        info["df"].show(5, truncate=False)
        print()
        continue

    df = info["df"]
    for col_json in info["colunas_json"]:
        print(f"--- {nome} | Coluna: {col_json} ---")
        amostra = df.select(col_json).filter(df[col_json].isNotNull()).first()[0]
        json_schema = schema_of_json(amostra)

        df_parsed = df.withColumn("parsed", from_json(df[col_json], json_schema)) \
                      .select("parsed.*")

        colunas_antes = set(df_parsed.columns)

        # Parsear sub-JSONs e achatar tudo
        df_flat = parse_nested_json(df_parsed)

        colunas_expandidas = [c for c in df_flat.columns if c not in colunas_antes]
        if colunas_expandidas:
            print(f"Colunas JSON aninhadas expandidas: {colunas_expandidas}")

        print(f"Total de colunas: {len(df_flat.columns)}")
        df_flat.printSchema()
        df_flat.show(5, truncate=False)

        info["df_parsed"] = df_flat
    print()

--- Customer_20260207170555.parquet: Sem JSON, mostrando dados diretos ---
+-------------+
|raw_value    |
+-------------+
|QueryResponse|
|time         |
+-------------+


--- Customer_20260325043100.parquet | Coluna: raw_value ---
Colunas JSON aninhadas expandidas: ['AlternatePhone_FreeFormNumber', 'BillAddr_City', 'BillAddr_CountrySubDivisionCode', 'BillAddr_Line1', 'BillAddr_PostalCode', 'CurrencyRef_name', 'CurrencyRef_value', 'DefaultTaxCodeRef_name', 'DefaultTaxCodeRef_value', 'MetaData_CreateTime', 'MetaData_LastUpdatedTime', 'MetaData_source', 'Mobile_FreeFormNumber', 'PaymentMethodRef_name', 'PaymentMethodRef_value', 'PrimaryEmailAddr_Address', 'PrimaryPhone_FreeFormNumber', 'SalesTermRef_name', 'SalesTermRef_value', 'ShipAddr_City', 'ShipAddr_CountrySubDivisionCode', 'ShipAddr_Line1', 'ShipAddr_PostalCode', 'WebAddr_URI']
Total de colunas: 46
root
 |-- Active: string (nullable = true)
 |-- AlternatePhone_FreeFormNumber: string (nullable = true)
 |-- Balance: string (nullable

In [11]:
from IPython.display import display
from functools import reduce
from pyspark.sql.functions import col, count, when

# Juntar todos os DataFrames parseados em um só, tratando diferenças de schema
dfs_parsed = [info["df_parsed"] for info in dataframes.values() if "df_parsed" in info]

if dfs_parsed:
    df_merged = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_parsed)

    # Identificar e remover colunas que são totalmente nulas
    total = df_merged.count()
    nulls_por_coluna = df_merged.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in df_merged.columns]
    ).first()

    colunas_nulas = [c for c in df_merged.columns if nulls_por_coluna[c] == total]
    if colunas_nulas:
        print(f"Colunas totalmente nulas removidas: {colunas_nulas}")
        df_merged = df_merged.drop(*colunas_nulas)

    print(f"Total de linhas: {total} | Total de colunas: {len(df_merged.columns)}")
    display(df_merged.limit(10).toPandas())
else:
    print("Nenhum DataFrame parseado encontrado.")

Colunas totalmente nulas removidas: ['Fax', 'ParentRef', 'Suffix', 'AlternatePhone', 'BillAddr', 'CurrencyRef', 'DefaultTaxCodeRef', 'MetaData', 'Mobile', 'PaymentMethodRef', 'PrimaryEmailAddr', 'PrimaryPhone', 'SalesTermRef', 'ShipAddr', 'WebAddr']
Total de linhas: 11 | Total de colunas: 43


,Active,AlternatePhone_FreeFormNumber,Balance,BalanceWithJobs,BillAddr_City,BillAddr_CountrySubDivisionCode,BillAddr_Line1,BillAddr_PostalCode,BillWithParent,CompanyName,...,ShipAddr_City,ShipAddr_CountrySubDivisionCode,ShipAddr_Line1,ShipAddr_PostalCode,SyncToken,Taxable,Title,WebAddr_URI,domain,sparse
0,true,555-21001,25.00,25.00,Sylacauga,AL,101 Main St,35151,false,Family Account 1,...,Sylacauga,AL,101 Main St,35151,301,false,Mr.,https://example.com/patient1,QBO,false
1,true,555-21002,50.00,50.00,Sylacauga,AL,102 Main St,35152,false,Family Account 2,...,Sylacauga,AL,102 Main St,35152,302,false,Ms.,https://example.com/patient2,QBO,false
2,true,555-21003,75.00,75.00,Sylacauga,AL,103 Main St,35153,false,Family Account 3,...,Sylacauga,AL,103 Main St,35153,303,false,Mr.,https://example.com/patient3,QBO,false
3,true,555-21004,100.00,100.00,Sylacauga,AL,104 Main St,35154,false,Family Account 4,...,Sylacauga,AL,104 Main St,35154,304,false,Ms.,https://example.com/patient4,QBO,false
4,true,555-21005,125.00,125.00,Sylacauga,AL,105 Main St,35155,false,Family Account 5,...,Sylacauga,AL,105 Main St,35155,305,false,Mr.,https://example.com/patient5,QBO,false
5,true,555-21006,150.00,150.00,Sylacauga,AL,106 Main St,35156,false,Family Account 6,...,Sylacauga,AL,106 Main St,35156,306,false,Ms.,https://example.com/patient6,QBO,false
6,true,555-21007,175.00,175.00,Sylacauga,AL,107 Main St,35157,false,Family Account 7,...,Sylacauga,AL,107 Main St,35157,307,false,Mr.,https://example.com/patient7,QBO,false
7,true,555-21008,200.00,200.00,Sylacauga,AL,108 Main St,35158,false,Family Account 8,...,Sylacauga,AL,108 Main St,35158,308,false,Ms.,https://example.com/patient8,QBO,false
8,true,555-21009,225.00,225.00,Sylacauga,AL,109 Main St,35159,false,Family Account 9,...,Sylacauga,AL,109 Main St,35159,309,false,Mr.,https://example.com/patient9,QBO,false
9,true,555-21010,250.00,250.00,Sylacauga,AL,110 Main St,351510,false,Family Account 10,...,Sylacauga,AL,110 Main St,351510,310,false,Ms.,https://example.com/patient10,QBO,false


In [12]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
